In [17]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [18]:
df = pd.read_csv('datos.csv', sep=';')

Debido al uso de python y el formato de csv, dentro del dataset python no pudo reconocer ningun duplicado exacto
Luego dentro de la limpieza aplico una euristica para manejar los duplicados

In [19]:
# Transformacion del dataset
def convertir_bool(valor):
    if isinstance(valor, str):
        valor = valor.strip().lower()
        return valor in ['yes', 'y','true','1', 'si']
    return bool(valor)
df['Multiplataforma'] = df['Multiplataforma'].apply(convertir_bool)
df['Multiplataforma'] = df['Multiplataforma'].map({True: 'Sí', False: 'No'})
print(df['Multiplataforma'].head())

0    No
1    Sí
2    No
3    Sí
4    Sí
Name: Multiplataforma, dtype: object


In [20]:
#Transformacion del dataset
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
df['id'] = df['id'].astype(int)
df['multiplataforma'] = df['multiplataforma'].astype(bool)
text = ['titulo', 'genero','plataforma','fecha_lanzamiento','desarrollador']

df['genero'] = df['genero'].str.strip().str.lower().str.capitalize()

df['fecha_lanzamiento'] = pd.to_datetime(df['fecha_lanzamiento'], errors='coerce', dayfirst=True)
df['fecha_lanzamiento'] = df['fecha_lanzamiento'].dt.strftime('%Y-%m-%d')

for i in text:
    df[i] = df[i].astype('string')

In [ ]:
#formato de titulo
df['titulo'] = df['titulo'].str.title()
df['desarrollador'] = df['desarrollador'].str.title()

#Formato de genero
df['genero'] = df['genero'].str.replace(';', ',').str.replace('/', ',')

#Procesar precio para que lo tome como 0.0 y no null
df['precio_usd'] = df['precio_usd'].replace(['Gratis', 'Free', 'gratis', 'free'], 0.0)



In [ ]:
# Transformar los datos numericos
numeric_cols = ['puntuacion', 'precio_usd', 'metascore']
for i in numeric_cols:
    df[i] = pd.to_numeric(df[i], errors='coerce')

In [22]:
# para el punto de revisar cuales estan nulos
print(df[(df['metascore'] < 0) | (df['metascore'] > 100) | df['metascore'].isnull()])

    id         titulo      genero plataforma fecha_lanzamiento  puntuacion  \
5    6  The Witcher 3         Rpg        PS4              <NA>      2015.0   
8    9           <NA>  Estrategia         PC        2020-07-14         8.2   
28  29           <NA>    Strategy         PC        2020-07-14         8.2   
45  46  The Witcher 3         Rpg        PS4              <NA>      2015.0   
48  49           <NA>  Estrategia         PC        2020-07-14         8.2   
68  69           <NA>    Strategy         PC        2020-07-14         8.2   
85  86  The Witcher 3         Rpg        PS4              <NA>      2015.0   
88  89           <NA>  Estrategia         PC        2020-07-14         8.2   

   desarrollador  precio_usd  multiplataforma  metascore  unnamed:_10  
5            9.7         NaN             True        NaN         92.0  
8           <NA>       19.99             True        NaN          NaN  
28          <NA>       19.99             True        NaN          NaN  
45       

In [23]:
print(df[(df['precio_usd'] < 0) | df['precio_usd'].isnull()])

    id         titulo         genero plataforma fecha_lanzamiento  puntuacion  \
5    6  The Witcher 3            Rpg        PS4              <NA>      2015.0   
15  16    Overwatch 2        Shooter         PC        2022-10-04         6.5   
25  26  The Witcher 3  Rpg, aventura        PS4        2015-05-19         9.7   
35  36      Overwatch        Shooter         PC        2022-10-04         6.5   
45  46  The Witcher 3            Rpg        PS4              <NA>      2015.0   
55  56    Overwatch 2        Shooter         PC        2022-10-04         6.5   
65  66  The Witcher 3  Rpg, aventura        PS4        2015-05-19         9.7   
75  76      Overwatch        Shooter         PC        2022-10-04         6.5   
85  86  The Witcher 3            Rpg        PS4              <NA>      2015.0   
95  96    Overwatch 2        Shooter         PC        2022-10-04         6.5   

             desarrollador  precio_usd  multiplataforma  metascore  \
5                      9.7         NaN

In [24]:
df.drop(columns=['unnamed:_10'], inplace=True, errors='ignore')
df = df[df['titulo'].notna()]
df['plataforma'] = df['plataforma'].fillna(df['plataforma'].mode()[0])
df['desarrollador'] = df['desarrollador'].fillna('Desconocido')

df['precio_usd'] = df['precio_usd'].fillna(df['precio_usd'].median())
df['metascore'] = df['metascore'].fillna(df['metascore'].mean())


In [25]:
# vamos a procesar los duplicados
print('Antes: ')
print(df['titulo'].str.lower().duplicated().sum())
print('\n')
df['titulo_minuscula'] = df['titulo'].str.lower()

df.drop_duplicates(subset='titulo_minuscula', keep='first', inplace=True)

df.drop(columns='titulo_minuscula', inplace=True)
print('Despues: ')
print(df['titulo'].str.lower().duplicated().sum())

Antes: 
61


Despues: 
0


In [26]:
# puntuacion entre 0 y 10
df['puntuacion'] = df['puntuacion'].clip(lower=0,upper=10)

df = df[~df['desarrollador'].astype(str).str.contains(r'\d')]

# Validar
print(df['puntuacion'].describe())
print(df['desarrollador'].unique())

count    21.000000
mean      8.561905
std       1.086497
min       6.500000
25%       8.000000
50%       8.500000
75%       9.500000
max      10.000000
Name: puntuacion, dtype: float64
<StringArray>
[              'Nintendo',           'Santa Monica',                 'Mojang',
             'Epic Games',               'Rockstar',            'Naughty Dog',
             'Innersloth',         'Rockstar Games',          'Infinity Ward',
 'Blizzard Entertainment',           'Fromsoftware',           'Ea Vancouver',
         'Cd Projekt Red',         'Rockstar North']
Length: 14, dtype: string


In [27]:
df['metascore'] = df['metascore'].clip(lower=0, upper=100)


In [28]:
df.head(100)

,id,titulo,genero,plataforma,fecha_lanzamiento,puntuacion,desarrollador,precio_usd,multiplataforma,metascore
0,1,The Legend Of Zelda,Aventura,Switch,2017-03-03,9.5,Nintendo,59.99,True,97.0
1,2,God Of War,Acción,PS4,2018-04-20,9.8,Santa Monica,39.99,True,94.0
4,5,Minecraft,Sandbox,PS4,<NA>,9.0,Mojang,26.95,True,93.0
6,7,Fortnite,Battle royale,Multi,2017-07-25,8.0,Epic Games,0.00,True,78.0
7,8,Gta V,Acción,PS4,2013-09-17,10.0,Rockstar,29.99,True,97.0
9,10,Animal Crossing,Simulación,Switch,2020-03-20,8.5,Nintendo,49.99,True,90.0
10,11,The Last Of Us,"Acción, aventura",PS4,2020-06-19,8.5,Naughty Dog,49.99,True,93.0
12,13,Among Us,Party game,Mobile,2018-06-15,7.5,Innersloth,0.00,True,85.0
13,14,Red Dead Redemption 2,Aventura,PS4,2018-10-26,9.5,Rockstar Games,59.99,True,97.0
14,15,Call Of Duty,Fps,Multiplatform,2020-03-10,8.0,Infinity Ward,0.00,True,79.0


In [29]:
df.to_csv('datosClean.csv', index=False)